In [0]:
"""
04_material_events.py

Creates the Silver Material Events table.

Input:
    parsed_events

Output:
    material_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import col


# ============================================================
# Material Events
# ============================================================

@dp.table(
    name="material_events",
    comment="Validated manufacturing material scan events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_serial_number",
    "serial_number IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_material_number",
    "material_number IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_supplier",
    "supplier IS NOT NULL",
)

@dp.expect(
    "valid_scan_status",
    "scan_status IN ('SUCCESS', 'FAILED')",
)

def material_events():

    df = dp.read_stream("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only material scan events
        # -----------------------------------------

        .filter(
            col("event_type") == "MATERIAL_SCANNED"
        )

        # -----------------------------------------
        # Business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "execution_id",
            "serial_number",

            "product_code",

            "source_system",
            "correlation_id",

            "bronze_ingestion_timestamp",
            "silver_processing_timestamp",

            "payload.scan_id",

            "payload.material_number",
            "payload.batch_number",

            "payload.supplier",

            "payload.scan_status",

            "payload.product_name",
            "payload.family",

        )

    )